## EDA for text

Cell cluster distribution and class imbalance<br>
UMAP visualization<br>
Spatial coordinate distribution<br>
Transcript quantity (data quality)<br>
Marker gene heatmap<br>

In [1]:
# Breast Cancer Xenium Data - EDA
# Run on Kaggle: attach your dataset and update BASE_PATH below

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
import os
import warnings
warnings.filterwarnings('ignore')

In [2]:
# SECTION 0: CONFIG
import os

EXCEL_PATH           = '/kaggle/input/datasets/eva233333/data3888-grp/41467_2023_43458_MOESM4_ESM.xlsx'
CBR_PATH             = '/kaggle/input/datasets/eva233333/data3888-grp/cbr.csv'
CELL_BOUNDARIES_PATH = '/kaggle/input/datasets/eva233333/data3888-grp/cell_boundaries.csv'
THREEPTS_PATH        = '/kaggle/input/datasets/eva233333/data3888-grp/threepts.csv'
HISTOLOGY_PATH       = '/kaggle/input/datasets/eva233333/data3888-grp/GSM7780153_Post-Xenium_HE_Rep1.ome.tif'

# 100px image path
IMAGE_DIRS_100 = {
    'DCIS_1':                '/kaggle/input/datasets/eva233333/tumor-cell-dcis-2/DCIS_1/DCIS_1/',
    'DCIS_2':                '/kaggle/input/datasets/eva233333/tumor-cell-dcis-2/DCIS_2/DCIS_2/',
    'Prolif_Invasive_Tumor': '/kaggle/input/datasets/eva233333/tumor-cell-dcis-2/Prolif_Invasive_Tumor/Prolif_Invasive_Tumor/',
    'Invasive_Tumor':        '/kaggle/input/datasets/eva233333/tumor-cell-dcis-2/Invasive_Tumor/Invasive_Tumor/',
}

# 50px image path
IMAGE_DIRS_50 = {
    'DCIS_1':                '/kaggle/input/datasets/eva233333/50px-tumor-cell/DCIS_1/DCIS_1/',
    'DCIS_2':                '/kaggle/input/datasets/eva233333/50px-tumor-cell/DCIS_2/DCIS_2/',
    'Prolif_Invasive_Tumor': '/kaggle/input/datasets/eva233333/50px-tumor-cell/Prolif_Invasive_Tumor (1)/Prolif_Invasive_Tumor/',
    'Invasive_Tumor':        '/kaggle/input/datasets/eva233333/50px-tumor-cell/Invasive_Tumor (1)/Invasive_Tumor/',
}

LABEL_MAP = {
    'DCIS_1': 0,
    'DCIS_2': 1,
    'Prolif_Invasive_Tumor': 2,
    'Invasive_Tumor': 3,
}

TUMOR_TYPES_XENIUM = ['DCIS 1', 'DCIS 2', 'Prolif_Invasive_Tumor', 'Invasive_Tumor']

COLORS = {
    'DCIS_1':                '#E63946',
    'DCIS_2':                '#F4A261',
    'Prolif_Invasive_Tumor': '#2A9D8F',
    'Invasive_Tumor':        '#457B9D',
}

OUT_DIR = '/kaggle/working/'

def get_cell_id(filename):
    """'cell_158510_100.png' -> 158510"""
    return int(filename.split('_')[1])


print("\n 100px File number verification")
for cl, path in IMAGE_DIRS_100.items():
    n = len(os.listdir(path)) if os.path.exists(path) else "PATH NOT FOUND"
    print(f"  {cl:30s}: {n}")

print("\n 50px File number verification")
for cl, path in IMAGE_DIRS_50.items():
    n = len(os.listdir(path)) if os.path.exists(path) else "PATH NOT FOUND"
    print(f"  {cl:30s}: {n}")


 100px File number verification
  DCIS_1                        : 12925
  DCIS_2                        : 11719
  Prolif_Invasive_Tumor         : 3775
  Invasive_Tumor                : 34398

 50px File number verification
  DCIS_1                        : 12923
  DCIS_2                        : 11683
  Prolif_Invasive_Tumor         : 3775
  Invasive_Tumor                : 34374


In [3]:
# SECTION 1: LOAD DATA

xl = pd.read_excel(EXCEL_PATH, sheet_name=None)
print(f"Sheets found: {list(xl.keys())}")

xenium   = xl['Fig. 3e-j Xenium']        # main single-cell data
scffpe   = xl['Fig. 2a scFFPE-seq UMAP'] # scRNA-seq reference
heatmap  = xl['Fig. 3k Heatmap']         # marker gene expression

print(f"\nXenium shape:  {xenium.shape}")
print(f"scFFPE shape:  {scffpe.shape}")
print(f"Heatmap shape: {heatmap.shape}")
print("\nXenium columns:", list(xenium.columns))
print("\nXenium dtypes:\n", xenium.dtypes)


Sheets found: ['Fig. 2a scFFPE-seq UMAP', 'Fig. 3e-j Xenium', 'Fig. 3k Heatmap', 'Fig. 4a ROIs', 'Fig. 4b-d ', 'Fig. 5 Spot Binned Xenium Data', 'Fig. 6e Heatmap', 'Fig. 6f Violins', 'Sup. Fig. 1 Panel Heatmap', 'Sup. Fig. 2', 'Sup. Fig. 3 Flex Heatmap', 'Sup. Fig. 5 Scatter', 'Sup Fig. 5 Xenium Rep 2', 'Sup. Fig. 7 SC Comparison', 'Sup. Fig. 10 Visium Deconv.']

Xenium shape:  (167780, 8)
scFFPE shape:  (27472, 4)
Heatmap shape: (313, 21)

Xenium columns: ['Barcode', 'UMAP_DIM1', 'UMAP_DIM2', 'Cluster', 'transcript_counts', 'x_centroid', 'y_centroid', 'gene_counts']

Xenium dtypes:
 Barcode                int64
UMAP_DIM1            float64
UMAP_DIM2            float64
Cluster               object
transcript_counts      int64
x_centroid           float64
y_centroid           float64
gene_counts            int64
dtype: object


In [4]:
# SECTION 2: BASIC STATS

print("\n All cluster counts")
print(xenium['Cluster'].value_counts().to_string())

print("\n Missing values in Xenium")
print(xenium.isnull().sum())

print("\n Coordinate ranges")
print(f"x_centroid: {xenium['x_centroid'].min():.1f} → {xenium['x_centroid'].max():.1f}")
print(f"y_centroid: {xenium['y_centroid'].min():.1f} → {xenium['y_centroid'].max():.1f}")

print("\n Quality metrics (all cells)")
print(xenium[['transcript_counts', 'gene_counts']].describe().round(1))

# Cells with 0 transcripts
zero_tx = xenium[xenium['transcript_counts'] == 0]
print(f"\nCells with 0 transcripts: {len(zero_tx)}")
print(zero_tx['Cluster'].value_counts())


 All cluster counts
Cluster
Stromal                    41422
Invasive_Tumor             34374
DCIS 1                     12923
DCIS 2                     11683
Macrophages_1              11174
Endothelial                 8931
Unlabeled                   8554
CD4+_T_Cells                8453
Myoepi_ACTA2+               7078
CD8+_T_Cells                6940
B_Cells                     4987
Prolif_Invasive_Tumor       3775
Myoepi_KRT15+               2860
Macrophages_2               1624
Perivascular-Like            847
Stromal_&_T_Cell_Hybrid      607
T_Cell_&_Tumor_Hybrid        589
IRF7+_DCs                    494
LAMP3+_DCs                   298
Mast_Cells                   167

 Missing values in Xenium
Barcode                 0
UMAP_DIM1            7478
UMAP_DIM2            7478
Cluster                 0
transcript_counts       0
x_centroid              0
y_centroid              0
gene_counts             0
dtype: int64

 Coordinate ranges
x_centroid: 2.1 → 7523.1
y_centroid: 1.4 → 

In [5]:
# SECTION 3: TUMOR CELL ANALYSIS
TUMOR_TYPES = ['DCIS 1', 'DCIS 2', 'Prolif_Invasive_Tumor', 'Invasive_Tumor']  # Xenium表里用空格

tumor_df = xenium[xenium['Cluster'].isin(TUMOR_TYPES)].copy()
counts   = tumor_df['Cluster'].value_counts()

print(f"\n Tumor cell counts")
print(counts)
print(f"\nTotal tumor cells: {len(tumor_df)}")
print(f"Class imbalance (max/min): {counts.max()/counts.min():.1f}x")

print("\n Transcript counts per tumor cluster")
print(tumor_df.groupby('Cluster')['transcript_counts'].describe().round(1))

print("\n Spatial extent per cluster")
for cl in TUMOR_TYPES:
    sub = tumor_df[tumor_df['Cluster'] == cl]
    print(f"{cl:30s}: n={len(sub):5d} | "
          f"x=[{sub.x_centroid.min():.0f},{sub.x_centroid.max():.0f}] "
          f"y=[{sub.y_centroid.min():.0f},{sub.y_centroid.max():.0f}]")

# Naming inconsistency warning
print("\n Cluster name formats (watch for space vs underscore!)")
for c in sorted(xenium['Cluster'].unique()):
    print(repr(c))




 Tumor cell counts
Cluster
Invasive_Tumor           34374
DCIS 1                   12923
DCIS 2                   11683
Prolif_Invasive_Tumor     3775
Name: count, dtype: int64

Total tumor cells: 62755
Class imbalance (max/min): 9.1x

 Transcript counts per tumor cluster
                         count   mean    std   min    25%    50%    75%  \
Cluster                                                                   
DCIS 1                 12923.0  284.0  132.4  11.0  189.0  271.0  364.0   
DCIS 2                 11683.0  216.7  128.8  11.0  125.0  200.0  285.0   
Invasive_Tumor         34374.0  232.0  129.7  11.0  138.0  214.0  305.0   
Prolif_Invasive_Tumor   3775.0  307.4  150.7  20.0  195.0  288.0  394.0   

                          max  
Cluster                        
DCIS 1                 1329.0  
DCIS 2                 1259.0  
Invasive_Tumor         1181.0  
Prolif_Invasive_Tumor  1029.0  

 Spatial extent per cluster
DCIS 1                        : n=12923 | x=[3,7521] y

In [6]:
# SECTION 4: IMAGE FILE EXPLORATION

for size_label, image_dirs in [('50px', IMAGE_DIRS_50), ('100px', IMAGE_DIRS_100)]:
    print(f"\n {size_label}")
    for cl, path in image_dirs.items():
        p = Path(path)
        if not p.exists():
            print(f"  WARNING: {path} not found")
            continue
        imgs = list(p.glob('*.png'))
        print(f"  {cl:30s}: {len(imgs):6d} images")
        if imgs:
            for img in imgs[:2]:
                print(f"    example: {img.name}")


 50px
  DCIS_1                        :  12923 images
    example: cell_138442_50.png
    example: cell_153762_50.png
  DCIS_2                        :  11683 images
    example: cell_38236_50.png
    example: cell_67373_50.png
  Prolif_Invasive_Tumor         :   3775 images
    example: cell_71291_50.png
    example: cell_64497_50.png
  Invasive_Tumor                :  34374 images
    example: cell_22488_50.png
    example: cell_73465_50.png

 100px
  DCIS_1                        :  12925 images
    example: cell_130874_100.png
    example: cell_161072_100.png
  DCIS_2                        :  11719 images
    example: cell_158510_100.png
    example: cell_46750_100.png
  Prolif_Invasive_Tumor         :   3775 images
    example: cell_59836_100.png
    example: cell_17266_100.png
  Invasive_Tumor                :  34398 images
    example: cell_24387_100.png
    example: cell_72695_100.png


In [7]:
# SECTION 5: BARCODE <-> FILENAME MATCHING

# Try to figure out if image filenames contain barcodes
import pandas as pd

xenium = pd.read_excel(EXCEL_PATH, sheet_name='Fig. 3e-j Xenium')
xenium['Cluster'] = xenium['Cluster'].str.replace(' ', '_')
xenium['cell_id'] = range(1, len(xenium) + 1)  # cell_id是1-based行号

print(f"Total number of cells in Xenium: {len(xenium)}")
print(f"cell_id range: {xenium['cell_id'].min()} - {xenium['cell_id'].max()}")

# Verify using the sample file of DCIS_1
sample_dir = IMAGE_DIRS_100['DCIS_1']
sample_files = os.listdir(sample_dir)[:10]

print("\n Verification result (cell_id -> Xenium coordinates):")
all_matched = True
for fname in sample_files:
    cid = get_cell_id(fname)
    row = xenium[xenium['cell_id'] == cid]
    if len(row) > 0:
        r = row.iloc[0]
        print(f"  {fname:30s} -> cluster={r['Cluster']:25s} x={r['x_centroid']:.1f} y={r['y_centroid']:.1f} ✓")
    else:
        print(f"  {fname:30s} -> NOT FOUND ✗")
        all_matched = False

Total number of cells in Xenium: 167780
cell_id range: 1 - 167780

 Verification result (cell_id -> Xenium coordinates):
  cell_130874_100.png            -> cluster=DCIS_1                    x=5119.4 y=3357.5 ✓
  cell_161072_100.png            -> cluster=DCIS_1                    x=7062.4 y=3457.1 ✓
  cell_133340_100.png            -> cluster=DCIS_1                    x=5385.7 y=3460.9 ✓
  cell_154812_100.png            -> cluster=DCIS_1                    x=7052.5 y=2132.1 ✓
  cell_167130_100.png            -> cluster=DCIS_1                    x=7452.2 y=3314.7 ✓
  cell_157898_100.png            -> cluster=DCIS_1                    x=7124.3 y=3022.4 ✓
  cell_153864_100.png            -> cluster=DCIS_1                    x=7246.3 y=2190.9 ✓
  cell_165906_100.png            -> cluster=DCIS_1                    x=7399.8 y=5102.9 ✓
  cell_13941_100.png             -> cluster=DCIS_1                    x=7510.7 y=3356.6 ✓
  cell_162275_100.png            -> cluster=DCIS_1                   

In [8]:
# SECTION 6: VISUALISATIONS
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Unified Cluster Naming (Spaces → Underscores)
xenium['Cluster'] = xenium['Cluster'].str.replace(' ', '_')

TUMOR_TYPES = ['DCIS_1', 'DCIS_2', 'Prolif_Invasive_Tumor', 'Invasive_Tumor']
COLORS = {
    'DCIS_1':                '#E63946',
    'DCIS_2':                '#F4A261',
    'Prolif_Invasive_Tumor': '#2A9D8F',
    'Invasive_Tumor':        '#457B9D',
}
tumor_df = xenium[xenium['Cluster'].isin(TUMOR_TYPES)].copy()
counts   = tumor_df['Cluster'].value_counts()
patches  = [mpatches.Patch(color=COLORS[c], label=c) for c in TUMOR_TYPES]

# Figure 1: Main EDA overview (6 panels) 
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('EDA: Xenium Breast Cancer Single-Cell Data', fontsize=16, fontweight='bold')

# Panel 1: All cluster sizes
ax = axes[0, 0]
all_counts  = xenium['Cluster'].value_counts()
bar_colors  = ['#E63946' if c in TUMOR_TYPES else '#AAAAAA' for c in all_counts.index]
ax.barh(range(len(all_counts)), all_counts.values, color=bar_colors)
ax.set_yticks(range(len(all_counts)))
ax.set_yticklabels(all_counts.index, fontsize=8)
ax.set_xlabel('Cell Count')
ax.set_title('All Cluster Sizes\n(red = selected tumor types)', fontweight='bold')
ax.invert_yaxis()

# Panel 2: Tumor class imbalance
ax = axes[0, 1]
bar_cols = [COLORS[c] for c in counts.index]
ax.bar(range(len(counts)), counts.values, color=bar_cols, edgecolor='white', width=0.6)
ax.set_xticks(range(len(counts)))
ax.set_xticklabels([c.replace('_', '\n') for c in counts.index], fontsize=9)
ax.set_ylabel('Cell Count')
ax.set_title('Tumor Cell Class Imbalance\n(9.1× max/min ratio)', fontweight='bold')
for i, v in enumerate(counts.values):
    ax.text(i, v + 200, f'{v:,}', ha='center', fontsize=9, fontweight='bold')

# Panel 3: UMAP — all cells, tumor highlighted
ax = axes[0, 2]
non_tumor = xenium[~xenium['Cluster'].isin(TUMOR_TYPES)].dropna(subset=['UMAP_DIM1'])
ax.scatter(non_tumor['UMAP_DIM1'], non_tumor['UMAP_DIM2'],
           c='#DDDDDD', s=0.3, alpha=0.3, rasterized=True)
for cl in TUMOR_TYPES:
    sub = xenium[xenium['Cluster'] == cl].dropna(subset=['UMAP_DIM1'])
    ax.scatter(sub['UMAP_DIM1'], sub['UMAP_DIM2'],
               c=COLORS[cl], s=0.5, alpha=0.6, label=cl, rasterized=True)
ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
ax.set_title('UMAP (Xenium)\nTumor clusters highlighted', fontweight='bold')
patches = [mpatches.Patch(color=COLORS[c], label=c) for c in TUMOR_TYPES]
ax.legend(handles=patches, fontsize=7, loc='lower right')

# Panel 4: Spatial map
ax = axes[1, 0]
non_t = xenium[~xenium['Cluster'].isin(TUMOR_TYPES)]
ax.scatter(non_t['x_centroid'], non_t['y_centroid'],
           c='#EEEEEE', s=0.05, alpha=0.2, rasterized=True)
for cl in TUMOR_TYPES:
    sub = xenium[xenium['Cluster'] == cl]
    ax.scatter(sub['x_centroid'], sub['y_centroid'],
               c=COLORS[cl], s=0.2, alpha=0.5, rasterized=True)
ax.set_xlabel('X centroid (µm)'); ax.set_ylabel('Y centroid (µm)')
ax.set_title('Spatial Map of Tumour Cells\n(grey = other cell types)', fontweight='bold')
ax.legend(handles=patches, fontsize=7)
ax.invert_yaxis()

# Panel 5: Transcript count distribution
ax = axes[1, 1]
for cl in TUMOR_TYPES:
    sub = tumor_df[tumor_df['Cluster'] == cl]['transcript_counts']
    ax.hist(sub, bins=60, alpha=0.5, color=COLORS[cl], label=cl, density=True)
ax.set_xlabel('Transcript Count'); ax.set_ylabel('Density')
ax.set_title('Transcript Count per Cluster\n(proxy for data quality)', fontweight='bold')
ax.legend(fontsize=7)
ax.set_xlim(0, 900)

# Panel 6: Marker gene heatmap
ax = axes[1, 2]
tumor_cols_hm = ['DCIS_1', 'DCIS_2', 'Prolif_Invasive_Tumor', 'Invasive_Tumor']
hm = heatmap[['Genes'] + tumor_cols_hm].set_index('Genes')
hm['_max'] = hm.abs().max(axis=1)
top15 = hm.nlargest(15, '_max').drop('_max', axis=1)
im = ax.imshow(top15.values.T, aspect='auto', cmap='RdBu_r', vmin=-2, vmax=2)
ax.set_xticks(range(len(top15.index)))
ax.set_xticklabels(top15.index, rotation=45, ha='right', fontsize=7)
ax.set_yticks(range(len(tumor_cols_hm)))
ax.set_yticklabels([c.replace('_', ' ') for c in tumor_cols_hm], fontsize=8)
ax.set_title('Top 15 Marker Genes\n(tumor clusters)', fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.7, label='Scaled Expression')

plt.tight_layout()
out1 = OUT_DIR + 'fig1_eda_overview.png'
plt.savefig(out1, dpi=150, bbox_inches='tight')
plt.close()


# Figure 2: UMAP side-by-side comparison (scFFPE vs Xenium) 
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('UMAP Comparison: scFFPE-seq vs Xenium', fontsize=14, fontweight='bold')

for ax, df, dim1, dim2, label_col, title in [
    (axes[0], scffpe,  'UMAP-X',    'UMAP-Y',    'Annotation', 'scFFPE-seq'),
    (axes[1], xenium,  'UMAP_DIM1', 'UMAP_DIM2', 'Cluster',    'Xenium'),
]:
    df_valid = df.dropna(subset=[dim1, dim2])
    non_t = df_valid[~df_valid[label_col].isin(TUMOR_TYPES)]
    ax.scatter(non_t[dim1], non_t[dim2], c='#DDDDDD', s=0.3, alpha=0.3, rasterized=True)
    for cl in TUMOR_TYPES:
        sub = df_valid[df_valid[label_col] == cl]
        if len(sub) > 0:
            ax.scatter(sub[dim1], sub[dim2], c=COLORS[cl], s=0.5, alpha=0.7,
                       label=cl, rasterized=True)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel(dim1); ax.set_ylabel(dim2)

axes[1].legend(handles=patches, fontsize=8, loc='lower right')
plt.tight_layout()
out2 = OUT_DIR + 'fig2_umap_comparison.png'
plt.savefig(out2, dpi=150, bbox_inches='tight')
plt.close()


# Figure 3: Spatial map — all 20 cluster types 
fig, ax = plt.subplots(figsize=(12, 9))
unique_clusters = xenium['Cluster'].unique()
cmap = plt.cm.get_cmap('tab20', len(unique_clusters))
cluster_color_map = {cl: cmap(i) for i, cl in enumerate(unique_clusters)}

for cl in unique_clusters:
    sub = xenium[xenium['Cluster'] == cl]
    size = 0.5 if cl in TUMOR_TYPES else 0.1
    alpha = 0.7 if cl in TUMOR_TYPES else 0.2
    ax.scatter(sub['x_centroid'], sub['y_centroid'],
               c=[cluster_color_map[cl]], s=size, alpha=alpha,
               label=cl, rasterized=True)

ax.set_xlabel('X centroid (µm)'); ax.set_ylabel('Y centroid (µm)')
ax.set_title('Full Tissue Spatial Map — All Cell Types', fontweight='bold')
ax.invert_yaxis()
ax.legend(markerscale=8, fontsize=6, loc='upper right',
          ncol=2, bbox_to_anchor=(1.25, 1))
plt.tight_layout()
out3 = OUT_DIR + 'fig3_spatial_all_clusters.png'
plt.savefig(out3, dpi=150, bbox_inches='tight', bbox_extra_artists=[ax.legend()])
plt.close()


# Figure 4: Nearest-neighbour composition (microenvironment preview)
from sklearn.neighbors import BallTree

coords = xenium[['x_centroid', 'y_centroid']].values
tree   = BallTree(np.radians(coords) if False else coords)  # Euclidean

# For each tumor cell, find k=10 neighbours and record their cluster composition
K = 10
tumor_idx = xenium[xenium['Cluster'].isin(TUMOR_TYPES)].index.tolist()
# Use a subsample for speed (5000 cells)
rng = np.random.default_rng(42)
sample_idx = rng.choice(tumor_idx, size=min(5000, len(tumor_idx)), replace=False)

all_clusters = xenium['Cluster'].values
neighbour_compositions = {cl: {t: 0 for t in TUMOR_TYPES} for cl in TUMOR_TYPES}

sample_pos = xenium.loc[sample_idx, ['x_centroid', 'y_centroid']].values
_, nn_indices = tree.query(sample_pos, k=K+1)  # +1 because cell itself is included

for i, idx in enumerate(sample_idx):
    focal_cluster = xenium.loc[idx, 'Cluster']
    neighbours    = nn_indices[i, 1:]  # exclude self
    neighbour_clusters = all_clusters[neighbours]
    for nc in neighbour_clusters:
        if nc in TUMOR_TYPES:
            neighbour_compositions[focal_cluster][nc] += 1

# Normalise to proportions
fig, ax = plt.subplots(figsize=(8, 5))
comp_df = pd.DataFrame(neighbour_compositions).T
comp_df = comp_df.div(comp_df.sum(axis=1), axis=0)
comp_df.plot(kind='bar', ax=ax, color=[COLORS[c] for c in comp_df.columns],
             edgecolor='white', width=0.7)
ax.set_xlabel('Focal Cell Cluster')
ax.set_ylabel('Proportion of Tumor Neighbours')
ax.set_title(f'Tumour Microenvironment Composition\n(k={K} nearest neighbours, n=5000 sample)',
             fontweight='bold')
ax.legend(title='Neighbour type', fontsize=8)
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right')
plt.tight_layout()
out4 = OUT_DIR + 'fig4_microenvironment.png'
plt.savefig(out4, dpi=150, bbox_inches='tight')
plt.close()




In [9]:
# SECTION 7: EXPORT CLEAN LABEL TABLE

clean = tumor_df[['Barcode', 'Cluster', 'x_centroid', 'y_centroid',
                   'transcript_counts', 'gene_counts']].copy()

# Standardise cluster names: replace space with underscore
clean['Cluster'] = clean['Cluster'].str.replace(' ', '_')

# Add numeric label for ML
label_map = {
    'DCIS_1': 0,
    'DCIS_2': 1,
    'Prolif_Invasive_Tumor': 2,
    'Invasive_Tumor': 3,
}
clean['label'] = clean['Cluster'].map(label_map)

# Basic quality filter: keep cells with at least 10 transcripts
before = len(clean)
clean  = clean[clean['transcript_counts'] >= 10]
print(f"  Filtered {before - len(clean)} low-quality cells (transcript_count < 10)")
print(f"  Final table: {len(clean)} cells")
print(clean.head())

out_csv = OUT_DIR + 'tumor_cells_clean.csv'
clean.to_csv(out_csv, index=False)


  Filtered 0 low-quality cells (transcript_count < 10)
  Final table: 62755 cells
   Barcode         Cluster  x_centroid  y_centroid  transcript_counts  \
0        1          DCIS_2  847.259912  326.191365                 28   
1        2          DCIS_2  826.341995  328.031830                 94   
3        4  Invasive_Tumor  824.228409  334.252643                 11   
4        5          DCIS_2  841.357538  332.242505                 48   
7        8          DCIS_2  828.726239  341.712347                 39   

   gene_counts  label  
0           15      1  
1           38      1  
3            9      3  
4           33      1  
7           25      1  


## EDA&Data Preprocessing for images

In [10]:
# IMAGE EDA + PREPROCESSING
import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from pathlib import Path
from collections import defaultdict
warnings.filterwarnings('ignore')

### PART 1 — BASIC EDA

In [11]:
# IMAGE_DIRS_100, IMAGE_DIRS_50, LABEL_MAP, COLORS, OUT_DIR, get_cell_id
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

#### 1-A  Sample counts & class distribution

In [12]:
records_100, records_50 = [], []

for cl, path in IMAGE_DIRS_100.items():
    files = [f for f in os.listdir(path) if f.endswith('.png')]
    for f in files:
        records_100.append({'cluster': cl, 'label': LABEL_MAP[cl],
                            'cell_id': get_cell_id(f), 'path': os.path.join(path, f)})

for cl, path in IMAGE_DIRS_50.items():
    files = [f for f in os.listdir(path) if f.endswith('.png')]
    for f in files:
        records_50.append({'cluster': cl, 'label': LABEL_MAP[cl],
                           'cell_id': get_cell_id(f), 'path': os.path.join(path, f)})

df100 = pd.DataFrame(records_100)
df50  = pd.DataFrame(records_50)

# Intersection (cells with both sizes)
ids_both = {}
for cl in IMAGE_DIRS_100.keys():
    ids_100 = set(df100[df100['cluster'] == cl]['cell_id'])
    ids_50  = set(df50[df50['cluster']  == cl]['cell_id'])
    ids_both[cl] = ids_100 & ids_50

df_both = pd.concat([
    df100[(df100['cluster'] == cl) & (df100['cell_id'].isin(ids))]
    for cl, ids in ids_both.items()
]).reset_index(drop=True)

counts_100  = df100['cluster'].value_counts().sort_index()
counts_50   = df50['cluster'].value_counts().sort_index()
counts_both = df_both['cluster'].value_counts().sort_index()

print(f"\n{'Cluster':<30} {'100px':>8} {'50px':>8} {'Both':>8}")
print("-" * 58)
for cl in sorted(IMAGE_DIRS_100.keys()):
    print(f"{cl:<30} {counts_100.get(cl,0):>8,} {counts_50.get(cl,0):>8,} {counts_both.get(cl,0):>8,}")
print(f"{'TOTAL':<30} {len(df100):>8,} {len(df50):>8,} {len(df_both):>8,}")

imbalance = counts_both.max() / counts_both.min()
print(f"\nClass imbalance ratio (max/min): {imbalance:.1f}x")

# Plot
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('1-A: Sample Counts & Class Distribution', fontweight='bold')

for ax, counts, title in zip(axes,
    [counts_100, counts_50, counts_both],
    ['100px', '50px', 'Intersection (both)']):
    bars = ax.bar(range(len(counts)), counts.values,
                  color=[COLORS[c] for c in counts.index], edgecolor='white')
    ax.set_xticks(range(len(counts)))
    ax.set_xticklabels([c.replace('_', '\n') for c in counts.index], fontsize=8)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Cell Count')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 100, f'{v:,}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig(OUT_DIR + 'basic_1a_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()



Cluster                           100px     50px     Both
----------------------------------------------------------
DCIS_1                           12,925   12,923   12,925
DCIS_2                           11,719   11,683   11,719
Invasive_Tumor                   34,398   34,374   34,398
Prolif_Invasive_Tumor             3,775    3,775    3,775
TOTAL                            62,817   62,755   62,817

Class imbalance ratio (max/min): 9.1x


#### # 1-B  Image size distribution (width, height, aspect ratio)

In [13]:
size_records = []
SAMPLE_N = 200  # sample per cluster for speed

for cl, path in IMAGE_DIRS_100.items():
    files = random.sample(os.listdir(path), min(SAMPLE_N, len(os.listdir(path))))
    for fname in files:
        img = Image.open(os.path.join(path, fname))
        w, h = img.size
        size_records.append({'cluster': cl, 'size': '100px', 'width': w, 'height': h,
                              'aspect': w / h})

for cl, path in IMAGE_DIRS_50.items():
    files = random.sample(os.listdir(path), min(SAMPLE_N, len(os.listdir(path))))
    for fname in files:
        img = Image.open(os.path.join(path, fname))
        w, h = img.size
        size_records.append({'cluster': cl, 'size': '50px', 'width': w, 'height': h,
                              'aspect': w / h})

size_df = pd.DataFrame(size_records)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('1-B: Image Size Distribution', fontweight='bold')

for ax, col, xlabel in zip(axes,
    ['width', 'height', 'aspect'],
    ['Width (px)', 'Height (px)', 'Aspect Ratio (W/H)']):
    for sz, color in [('50px', '#457B9D'), ('100px', '#E63946')]:
        vals = size_df[size_df['size'] == sz][col]
        ax.hist(vals, bins=30, alpha=0.6, label=sz, color=color, density=True)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Density')
    ax.set_title(col, fontweight='bold')
    ax.legend()

plt.tight_layout()
plt.savefig(OUT_DIR + 'basic_1b_image_sizes.png', dpi=150, bbox_inches='tight')
plt.show()


### PART 2 — DEEPER IMAGE EDA


#### 2-A  Tissue content (non-white pixel ratio)

In [14]:
# 2-A  Tissue content (non-white pixel ratio)
def tissue_ratio(img_path):
    img = np.array(Image.open(img_path).convert('RGB'))
    return 1 - np.all(img > 230, axis=-1).mean()

tissue_records = []

for cl, path in IMAGE_DIRS_100.items():
    files = random.sample(os.listdir(path), min(SAMPLE_N, len(os.listdir(path))))
    for fname in files:
        ratio = tissue_ratio(os.path.join(path, fname))
        tissue_records.append({'cluster': cl, 'size': '100px', 'tissue_ratio': ratio})

for cl, path in IMAGE_DIRS_50.items():
    files = random.sample(os.listdir(path), min(SAMPLE_N, len(os.listdir(path))))
    for fname in files:
        ratio = tissue_ratio(os.path.join(path, fname))
        tissue_records.append({'cluster': cl, 'size': '50px', 'tissue_ratio': ratio})

tissue_df = pd.DataFrame(tissue_records)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('2-A: Tissue Content Distribution per Cluster', fontweight='bold')

for ax, size_label in zip(axes, ['100px', '50px']):
    sub = tissue_df[tissue_df['size'] == size_label]
    for cl in IMAGE_DIRS_100.keys():
        vals = sub[sub['cluster'] == cl]['tissue_ratio']
        ax.hist(vals, bins=40, alpha=0.5, label=cl, density=True, color=COLORS[cl])
    ax.axvline(0.05, color='red', linestyle='--', label='threshold=0.05')
    ax.set_xlabel('Tissue Ratio')
    ax.set_ylabel('Density')
    ax.set_title(size_label, fontweight='bold')
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig(OUT_DIR + 'deep_2a_tissue_content.png', dpi=150, bbox_inches='tight')
plt.show()

# Print statistics of low-quality patches
print("\nLow quality patches (tissue_ratio < 0.05):")
for size_label in ['100px', '50px']:
    sub = tissue_df[tissue_df['size'] == size_label]
    n_low = (sub['tissue_ratio'] < 0.05).sum()
    print(f"  {size_label}: {n_low} / {len(sub)} ({100*n_low/len(sub):.1f}%)")



Low quality patches (tissue_ratio < 0.05):
  100px: 2 / 800 (0.2%)
  50px: 0 / 800 (0.0%)


The tissue ratios of all four types of cells were concentrated around 0.9-1.0, far exceeding the threshold of 0.05, indicating that most patches had abundant tissue content and the problem of blank patches was basically non-existent.

#### 2-B  50px vs 100px side-by-side (same cell)

In [15]:
pairs = []
for cl in IMAGE_DIRS_100.keys():
    shared = list(ids_both[cl])
    if shared:
        sampled = random.sample(shared, min(3, len(shared)))
        for cid in sampled:
            pairs.append((cl, cid))

fig, axes = plt.subplots(len(pairs), 2, figsize=(5, len(pairs) * 2.2))
fig.suptitle('2-B: Same Cell — 50px vs 100px', fontweight='bold')
if len(pairs) == 1:
    axes = [axes]

for i, (cl, cid) in enumerate(pairs):
    p50  = os.path.join(IMAGE_DIRS_50[cl],  f'cell_{cid}_50.png')
    p100 = os.path.join(IMAGE_DIRS_100[cl], f'cell_{cid}_100.png')
    axes[i][0].imshow(Image.open(p50));  axes[i][0].axis('off')
    axes[i][1].imshow(Image.open(p100)); axes[i][1].axis('off')
    axes[i][0].set_ylabel(f'{cl}\ncell {cid}', fontsize=7,
                           rotation=0, labelpad=80, va='center')
    if i == 0:
        axes[i][0].set_title('50px', fontweight='bold')
        axes[i][1].set_title('100px', fontweight='bold')

plt.tight_layout()
plt.savefig(OUT_DIR + 'deep_2b_50vs100.png', dpi=150, bbox_inches='tight')
plt.show()


#### 2-C  Random sample grid (8 per cluster, 100px)

In [16]:
fig, axes = plt.subplots(4, 8, figsize=(20, 10))
fig.suptitle('2-C: Random Sample — 8 patches per cluster (100px)', fontweight='bold')

for row, (cl, path) in enumerate(IMAGE_DIRS_100.items()):
    files = random.sample(os.listdir(path), 8)
    for col, fname in enumerate(files):
        img = Image.open(os.path.join(path, fname))
        axes[row, col].imshow(img)
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_title(cl.replace('_', '\n'), 
                                      fontsize=9, fontweight='bold',
                                      color=COLORS[cl], loc='left')

plt.tight_layout()
plt.savefig(OUT_DIR + 'deep_2c_sample_grid.png', dpi=150, bbox_inches='tight')
plt.show()

### PART 3 — PREPROCESSING FOR CNN / ViT / GNN

#### 3-A  Compute dataset-level mean & std (for normalisation)

In [17]:
# Required by CNN and ViT
pixel_sum   = np.zeros(3)
pixel_sq    = np.zeros(3)
pixel_count = 0
NORM_SAMPLE = 1000  # increase for more accuracy, decrease for speed

all_paths_100 = []
for cl, path in IMAGE_DIRS_100.items():
    for f in os.listdir(path):
        if f.endswith('.png'):
            all_paths_100.append(os.path.join(path, f))

sample_paths = random.sample(all_paths_100, min(NORM_SAMPLE, len(all_paths_100)))

for fpath in sample_paths:
    img = np.array(Image.open(fpath).convert('RGB')) / 255.0  # H x W x 3
    pixel_sum   += img.sum(axis=(0, 1))
    pixel_sq    += (img ** 2).sum(axis=(0, 1))
    pixel_count += img.shape[0] * img.shape[1]

mean = pixel_sum / pixel_count
std  = np.sqrt(pixel_sq / pixel_count - mean ** 2)

# Save for later use
norm_stats = {'mean': mean.tolist(), 'std': std.tolist()}
pd.DataFrame(norm_stats).to_csv(OUT_DIR + 'norm_stats.csv', index=False)


3-A：计算归一化参数
从所有100px图片里随机抽1000张，计算整个数据集的RGB三通道的mean和std，保存成norm_stats.csv。
这个参数用在模型训练的transforms.Normalize(mean, std)里，把像素值从0-255标准化到均值为0、标准差为1的范围，让模型训练更稳定。
用自己数据集算出来的mean/std比直接用ImageNet的默认值（[0.485, 0.456, 0.406]）更准确，因为H&E染色图像的颜色分布和ImageNet的自然图像差异很大。

#### 3-B  Build master label CSV with consistent split

In [18]:
from sklearn.model_selection import train_test_split

master_records = []
for cl in IMAGE_DIRS_100.keys():
    shared_ids = list(ids_both[cl])  # Use only the cells that come in both sizes
    for cid in shared_ids:
        master_records.append({
            'cell_id':  cid,
            'cluster':  cl,
            'label':    LABEL_MAP[cl],
            'path_100': os.path.join(IMAGE_DIRS_100[cl], f'cell_{cid}_100.png'),
            'path_50':  os.path.join(IMAGE_DIRS_50[cl],  f'cell_{cid}_50.png'),
        })

master_df = pd.DataFrame(master_records)

# Stratified split by cell_id（70/15/15）
train_df, temp_df = train_test_split(
    master_df, test_size=0.30, stratify=master_df['label'], random_state=SEED)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df['label'], random_state=SEED)

master_df['split'] = 'train'
master_df.loc[val_df.index,  'split'] = 'val'
master_df.loc[test_df.index, 'split'] = 'test'

# Verify the split distribution
print(f"\nTotal cells (intersection): {len(master_df)}")
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print("\nClass distribution per split:")
for split in ['train', 'val', 'test']:
    sub = master_df[master_df['split'] == split]
    print(f"\n  {split} (n={len(sub)}):")
    for cl, n in sub['cluster'].value_counts().items():
        pct = 100 * n / len(sub)
        print(f"    {cl:<30}: {n:>6,}  ({pct:.1f}%)")

# Save - A file contains two paths, split sharing
master_df.to_csv(OUT_DIR + 'master_labels.csv', index=False)

# Verification: The same cell_id exists in both paths
sample = master_df.sample(10, random_state=SEED)
all_ok = True
for _, row in sample.iterrows():
    ok50  = os.path.exists(row['path_50'])
    ok100 = os.path.exists(row['path_100'])
    if not (ok50 and ok100):
        print(f"  ✗ cell_{row['cell_id']}: 50px={ok50}, 100px={ok100}")
        all_ok = False



Total cells (intersection): 62755
Train: 43928 | Val: 9413 | Test: 9414

Class distribution per split:

  train (n=43928):
    Invasive_Tumor                : 24,062  (54.8%)
    DCIS_1                        :  9,046  (20.6%)
    DCIS_2                        :  8,178  (18.6%)
    Prolif_Invasive_Tumor         :  2,642  (6.0%)

  val (n=9413):
    Invasive_Tumor                :  5,156  (54.8%)
    DCIS_1                        :  1,938  (20.6%)
    DCIS_2                        :  1,752  (18.6%)
    Prolif_Invasive_Tumor         :    567  (6.0%)

  test (n=9414):
    Invasive_Tumor                :  5,156  (54.8%)
    DCIS_1                        :  1,939  (20.6%)
    DCIS_2                        :  1,753  (18.6%)
    Prolif_Invasive_Tumor         :    566  (6.0%)


3-B：构建主标签表
把所有细胞整理成一张表master_labels.csv，每行是一个细胞，包含：

cell_id
cluster名称和数字label
50px图片路径
100px图片路径
split（train/val/test）

split是按70/15/15的比例做分层抽样，保证每个split里四种cluster的比例一致。
这张表是后续所有建模的核心索引，CNN、ViT、GNN都从这里读取数据。

#### 3-C  Class weight computation (for imbalanced training)

In [19]:
from sklearn.utils.class_weight import compute_class_weight

train_labels = master_df[master_df['split'] == 'train']['label'].values
class_weights = compute_class_weight('balanced', classes=np.unique(train_labels),
                                     y=train_labels)
weight_dict = dict(zip(np.unique(train_labels), class_weights))

for cl, label in LABEL_MAP.items():
    print(f"  {cl:<30}: label={label}, weight={weight_dict[label]:.4f}")

# As a tensor-ready list (ordered by label 0,1,2,3)
weight_list = [weight_dict[i] for i in sorted(weight_dict.keys())]

pd.DataFrame({'label': list(weight_dict.keys()),
              'weight': list(weight_dict.values())}).to_csv(
    OUT_DIR + 'class_weights.csv', index=False)



  DCIS_1                        : label=0, weight=1.2140
  DCIS_2                        : label=1, weight=1.3429
  Prolif_Invasive_Tumor         : label=2, weight=4.1567
  Invasive_Tumor                : label=3, weight=0.4564


因为四种细胞数量差距达9.1倍（Invasive_Tumor最多，Prolif_Invasive_Tumor最少），直接训练模型会偏向多数类。
类别权重的计算逻辑是：样本越少的类别权重越高，让模型在计算loss时更重视少数类。保存成class_weights.csv，训练时放进CrossEntropyLoss。
这样少数类预测错误会产生更大的loss，迫使模型认真学习少数类的特征。

#### 3-D  GNN graph construction (spatial k-NN graph from coordinates)

In [20]:
import pandas as pd
from sklearn.neighbors import BallTree

# Load Xenium coordinates
xl     = pd.read_excel(EXCEL_PATH, sheet_name='Fig. 3e-j Xenium')
xenium = xl.copy()
xenium['Cluster']  = xenium['Cluster'].str.replace(' ', '_')
xenium['cell_id']  = range(1, len(xenium) + 1)

# Keep only cells in our master dataset
valid_ids  = set(master_df['cell_id'].tolist())
xenium_sub = xenium[xenium['cell_id'].isin(valid_ids)].copy().reset_index(drop=True)
xenium_sub = xenium_sub.merge(
    master_df[['cell_id', 'label', 'split']], on='cell_id', how='left')

print(f"Cells with coordinates: {len(xenium_sub)}")

# Build k-NN spatial graph
K = 6  # number of spatial neighbours per cell
coords = xenium_sub[['x_centroid', 'y_centroid']].values
tree   = BallTree(coords)
distances, indices = tree.query(coords, k=K + 1)  # +1 includes self

# Build edge list
edges = []
for i in range(len(xenium_sub)):
    for j_pos in range(1, K + 1):          # skip self (index 0)
        j    = indices[i, j_pos]
        dist = distances[i, j_pos]
        src_id = xenium_sub.iloc[i]['cell_id']
        tgt_id = xenium_sub.iloc[j]['cell_id']
        edges.append({'src_cell_id': src_id, 'tgt_cell_id': tgt_id, 'distance': dist})

edge_df = pd.DataFrame(edges)
print(f"Edges created: {len(edge_df):,} (K={K} neighbours per cell)")
print(f"Mean neighbour distance: {edge_df['distance'].mean():.2f} µm")

edge_df.to_csv(OUT_DIR + 'gnn_edges.csv', index=False)
xenium_sub[['cell_id', 'x_centroid', 'y_centroid',
            'Cluster', 'label', 'split']].to_csv(
    OUT_DIR + 'gnn_nodes.csv', index=False)

# Visualise a small subgraph
fig, ax = plt.subplots(figsize=(10, 8))
fig.suptitle(f'3-D: Spatial k-NN Graph (K={K}) — sample region', fontweight='bold')

# Show a small spatial window
x_min, x_max = 1000, 2000
y_min, y_max = 1000, 2000
sub_nodes = xenium_sub[
    (xenium_sub['x_centroid'].between(x_min, x_max)) &
    (xenium_sub['y_centroid'].between(y_min, y_max))
]

sub_ids = set(sub_nodes['cell_id'])
sub_edges = edge_df[
    edge_df['src_cell_id'].isin(sub_ids) & edge_df['tgt_cell_id'].isin(sub_ids)
]

# Draw edges
id_to_xy = xenium_sub.set_index('cell_id')[['x_centroid', 'y_centroid']].to_dict('index')
for _, row in sub_edges.iterrows():
    x0, y0 = id_to_xy[row['src_cell_id']].values()
    x1, y1 = id_to_xy[row['tgt_cell_id']].values()
    ax.plot([x0, x1], [y0, y1], 'gray', alpha=0.3, linewidth=0.5)

# Draw nodes
for cl in LABEL_MAP.keys():
    sub_cl = sub_nodes[sub_nodes['Cluster'] == cl]
    ax.scatter(sub_cl['x_centroid'], sub_cl['y_centroid'],
               c=COLORS[cl], s=30, label=cl, zorder=3, edgecolors='white', linewidths=0.3)

ax.set_xlabel('X centroid (µm)'); ax.set_ylabel('Y centroid (µm)')
ax.legend(fontsize=8); ax.invert_yaxis()
plt.tight_layout()
plt.savefig(OUT_DIR + 'deep_3d_gnn_graph.png', dpi=150, bbox_inches='tight')
plt.show()


Cells with coordinates: 62755
Edges created: 376,530 (K=6 neighbours per cell)
Mean neighbour distance: 14.65 µm


Read the x_centroid and y_centroid values for each cell from the Excel file of Xenium, which represent the physical location of each cell on the tissue section.<br>
For each cell, find the six nearest neighbors in space and connect them with an edge. This way, all the cells on the entire section form a graph:<br>
Nodes = Each cell<br>
Edges = Connections between adjacent cells in space<br>
K = 6 Each cell can have at most 6 edges<br>
Draw the nodes and edges within a small window of the slice (x = 1000 - 2000, y = 1000 - 2000). You can see what the graph looks like.<br>

### PART 4 — PYTORCH DATASET CLASSES (ready to use)

In [21]:
print("PART 4: PYTORCH DATASET CLASSES")

pytorch_code = '''
# ── Copy this into your modelling notebook ────────────────────────────────────

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np

# Load pre-computed stats
norm_stats   = pd.read_csv("/kaggle/working/norm_stats.csv")
MEAN         = norm_stats["mean"].tolist()
STD          = norm_stats["std"].tolist()
master_df    = pd.read_csv("/kaggle/working/master_labels.csv")

# ── Transforms ────────────────────────────────────────────────────────────────
def get_transforms(split, img_size=100):
    if split == "train":
        return transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(90),
            transforms.ColorJitter(brightness=0.2, contrast=0.2,
                                   saturation=0.1, hue=0.05),
            transforms.ToTensor(),
            transforms.Normalize(mean=MEAN, std=STD),
        ])
    else:
        return transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=MEAN, std=STD),
        ])

# ── CNN / ViT Dataset (image only) ───────────────────────────────────────────
class CellImageDataset(Dataset):
    def __init__(self, df, split, img_size=100, use_100px=True):
        self.df        = df[df["split"] == split].reset_index(drop=True)
        self.transform = get_transforms(split, img_size)
        self.path_col  = "path_100" if use_100px else "path_50"

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        img   = Image.open(row[self.path_col]).convert("RGB")
        img   = self.transform(img)
        label = torch.tensor(row["label"], dtype=torch.long)
        return img, label

# ── GNN Dataset (image features + graph) ─────────────────────────────────────
class CellGraphDataset:
    """
    Returns a PyG Data object for the entire graph.
    Node features = CNN embeddings (computed separately).
    Edge index    = from gnn_edges.csv.
    """
    def __init__(self, node_csv, edge_csv):
        from torch_geometric.data import Data

        nodes = pd.read_csv(node_csv)
        edges = pd.read_csv(edge_csv)

        # Map cell_id to 0-based index
        id_to_idx = {cid: i for i, cid in enumerate(nodes["cell_id"])}
        src = edges["src_cell_id"].map(id_to_idx).values
        tgt = edges["tgt_cell_id"].map(id_to_idx).values

        self.edge_index = torch.tensor(np.stack([src, tgt]), dtype=torch.long)
        self.y          = torch.tensor(nodes["label"].values, dtype=torch.long)
        self.split_mask = {
            s: torch.tensor(nodes["split"] == s, dtype=torch.bool)
            for s in ["train", "val", "test"]
        }
        self.node_ids   = nodes["cell_id"].values

    def get_data(self, node_features: torch.Tensor):
        from torch_geometric.data import Data
        return Data(x=node_features, edge_index=self.edge_index, y=self.y)

# ── DataLoaders ───────────────────────────────────────────────────────────────
class_weights = torch.tensor(
    pd.read_csv("/kaggle/working/class_weights.csv")
      .sort_values("label")["weight"].tolist(),
    dtype=torch.float
)

def get_dataloaders(use_100px=True, img_size=100, batch_size=64):
    train_ds = CellImageDataset(master_df, "train", img_size, use_100px)
    val_ds   = CellImageDataset(master_df, "val",   img_size, use_100px)
    test_ds  = CellImageDataset(master_df, "test",  img_size, use_100px)

    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=2),
        DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=2),
        DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=2),
    )

# Usage:
# train_loader, val_loader, test_loader = get_dataloaders(use_100px=True)
# criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
'''

print(pytorch_code)

# Save as a separate file
with open(OUT_DIR + 'dataset_classes.py', 'w') as f:
    f.write(pytorch_code)

PART 4: PYTORCH DATASET CLASSES

# ── Copy this into your modelling notebook ────────────────────────────────────

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np

# Load pre-computed stats
norm_stats   = pd.read_csv("/kaggle/working/norm_stats.csv")
MEAN         = norm_stats["mean"].tolist()
STD          = norm_stats["std"].tolist()
master_df    = pd.read_csv("/kaggle/working/master_labels.csv")

# ── Transforms ────────────────────────────────────────────────────────────────
def get_transforms(split, img_size=100):
    if split == "train":
        return transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(90),
            transforms.ColorJitter(brightness=0.2, contrast=0.2,
                               